# GridShift — Notebook 03: Policy Simulation & PPO RL Training
## Training and Benchmarking the Reinforcement Learning Scheduler

This notebook covers:
1. **Interactive PPO Training**: Instantiates the Gymnasium `SchedulingEnv` on real production jobs and trains a `MaskablePPO` policy agent directly in this notebook.
2. **Action Masking Enforcement**: Verifies that the agent is constrained by capacity limits, preventing invalid scheduling decisions.
3. **Checkpoint Persistence**: Saves trained model weights to `artifacts/rl/checkpoints/ppo_forecast_seed42.zip`.
4. **Fair Policy Benchmark**: Replays identical test workloads through FCFS, Backfilling, and the learned PPO policy.
5. **Comparative Metrics**: Analyzes Mean Wait Times, P95 Tail Latency, Cluster GPU Utilization, and Throughput.
6. **Ablation Study**: Evaluates the effect of conditioning PPO on the 30-minute demand forecast vs. scheduling without forecast context.


In [ ]:
import sys
import json
from pathlib import Path

# Add src to Python path for clean modular imports
project_root = Path("..").resolve() if Path(".").resolve().name == "notebooks" else Path(".").resolve()
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Import clean modular components from src/
from gridshift.data.schemas import CanonicalJob
from gridshift.simulator.cluster import Cluster
from gridshift.rl.env import SchedulingEnv
from gridshift.forecasting.predictor import ForecastPredictor
from gridshift.evaluation.experiment import run_policy_experiment
from gridshift.evaluation.comparator import compare_policies

from sb3_contrib import MaskablePPO

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (10, 5)

print("GridShift RL environment & simulation modules imported!")


## 1. Load Trace & Initialize Cluster Configuration
We load the real production trace and define the cluster topology:
* **Machines**: 10 nodes
* **Capacity per node**: 8 GPUs, 64 CPU cores, 256 GB RAM (80 GPUs cluster capacity)


In [ ]:
parquet_path = project_root / "data" / "processed" / "alibaba_gpu_trace_canonical.parquet"
df_all = pd.read_parquet(parquet_path)

# Convert records to CanonicalJob objects
all_jobs = [CanonicalJob(**row) for row in df_all.to_dict("records")]
print(f"Loaded {len(all_jobs):,} canonical jobs.")

cluster_config = {
    "num_machines": 10,
    "gpus_per_machine": 8,
    "cpu_per_machine": 64.0,
    "memory_per_machine": 256.0,
}

cluster = Cluster(
    num_machines=cluster_config["num_machines"],
    gpus_per_machine=cluster_config["gpus_per_machine"],
    cpu_per_machine=cluster_config["cpu_per_machine"],
    memory_per_machine=cluster_config["memory_per_machine"],
)
print(f"Cluster: {len(cluster.machines)} machines, {cluster.total_gpus} total GPUs")


## 2. Interactive PPO Training Setup
We instantiate the custom Gymnasium `SchedulingEnv` ([`src/gridshift/rl/env.py`](file:///Users/vipulsharma/Desktop/GridShift/src/gridshift/rl/env.py)):
* **Observation**: Cluster node allocations + queue candidate features + 30-min demand forecast.
* **Action Space**: Discrete choice of $K=10$ candidate jobs, plus `WAIT`.
* **Action Masking**: Prevents any choice exceeding available cluster machine resources.


In [ ]:
forecast_model_path = project_root / "artifacts" / "forecasting" / "models" / "xgboost_forecast.json"
predictor = ForecastPredictor(model_path=str(forecast_model_path)) if forecast_model_path.exists() else None

# Initialize Gymnasium environment
train_jobs = all_jobs[:10000]  # Training slice
env = SchedulingEnv(
    jobs=train_jobs,
    cluster_config=cluster_config,
    candidate_k=10,
    forecast_predictor=predictor,
    seed=42,
)

print("Gymnasium SchedulingEnv initialized!")
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")


## 3. Train the MaskablePPO Agent Interactively
We train the PPO agent directly in this notebook cell using `sb3_contrib.MaskablePPO`.


In [ ]:
total_timesteps = 10000
seed = 42

model = MaskablePPO(
    "MlpPolicy",
    env,
    learning_rate=3e-4,
    n_steps=min(2048, len(train_jobs)),
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    verbose=1,
    seed=seed,
)

print(f"Starting PPO training for {total_timesteps:,} timesteps...")
model.learn(total_timesteps=total_timesteps, progress_bar=True)
print("PPO training finished!")


## 4. Save Trained Checkpoint to `artifacts/`
We persist the model weights and configuration to the artifacts directory.


In [ ]:
checkpoint_dir = project_root / "artifacts" / "rl" / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = checkpoint_dir / "ppo_forecast_seed42"

model.save(str(checkpoint_path))

config_payload = {
    "dataset": str(parquet_path),
    "cluster_config": cluster_config,
    "candidate_k": 10,
    "forecast_enabled": True,
    "seed": seed,
    "total_timesteps": total_timesteps,
    "checkpoint_path": f"{checkpoint_path}.zip",
}

with open(f"{checkpoint_path}_config.json", "w") as f:
    json.dump(config_payload, f, indent=2)

print(f"PPO checkpoint saved to: {checkpoint_path}.zip")
print(f"Config saved to: {checkpoint_path}_config.json")


## 5. Fair Policy Benchmark: FCFS vs. Backfilling vs. Trained PPO
We evaluate the newly trained PPO agent against traditional baselines on identical test jobs.


In [ ]:
# Test evaluation slice (held-out jobs)
test_jobs = all_jobs[10000:11000]
print(f"Benchmarking policies on {len(test_jobs):,} held-out test jobs...")

comparison = compare_policies(
    test_jobs=test_jobs,
    cluster_config=cluster_config,
    ppo_checkpoint=str(checkpoint_path) + ".zip",
    seed=42,
)

results_df = pd.DataFrame(comparison["results"])
display(results_df[["policy", "avg_wait_sec", "p95_wait_sec", "gpu_utilization", "throughput_jobs_hr", "constraint_violations", "completed_jobs"]])


## 6. Policy Comparison Visualizations
We plot comparative charts for latency, tail delays, utilization, and throughput.


In [ ]:
policies = [r["policy"].upper() for r in comparison["results"]]
avg_waits = [r["avg_wait_sec"] for r in comparison["results"]]
p95_waits = [r["p95_wait_sec"] for r in comparison["results"]]
utils = [r["gpu_utilization"] * 100 for r in comparison["results"]]
throughputs = [r["throughput_jobs_hr"] for r in comparison["results"]]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))

# Latency
x = np.arange(len(policies))
width = 0.35
ax1.bar(x - width/2, avg_waits, width, label="Mean Wait (s)", color="#0f172a")
ax1.bar(x + width/2, p95_waits, width, label="P95 Tail Wait (s)", color="#64748b")
ax1.set_title("Queue Latency Comparison (Lower is Better)", fontweight="bold")
ax1.set_xticks(x)
ax1.set_xticklabels(policies)
ax1.set_ylabel("Wait Time (seconds)")
ax1.legend()

# Cluster GPU Utilization
ax2.bar(policies, utils, color="#1e293b", width=0.5)
ax2.set_title("Average Cluster GPU Utilization (%)", fontweight="bold")
ax2.set_ylabel("Active GPU %")
ax2.set_ylim(0, 100)
for i, v in enumerate(utils):
    ax2.text(i, v + 2, f"{v:.1f}%", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()


## 7. Capacity Safety Verification
We confirm that action masking guarantees zero capacity exceptions throughout the test:


In [ ]:
print("=== CAPACITY INVARIANT AUDIT ===")
for r in comparison["results"]:
    p_name = r["policy"].upper()
    v_count = r["constraint_violations"]
    status = "SAFE (Passed)" if v_count == 0 else f"FAILED ({v_count} violations)"
    print(f"Policy: {p_name:<12} | Violations: {v_count} | Status: {status}")


## 8. Forecast Ablation Analysis
We compare PPO with forecast conditioning vs. PPO without forecast conditioning.


In [ ]:
ablation_checkpoint = project_root / "artifacts" / "rl" / "checkpoints" / "ppo_noforecast_seed42.zip"

if ablation_checkpoint.exists():
    res_noforecast = run_policy_experiment(
        jobs=test_jobs,
        policy="ppo",
        cluster_config=cluster_config,
        ppo_checkpoint=str(ablation_checkpoint),
        seed=42,
    )
    print("Ablation Run Completed!")
    print(f"PPO (Without Forecast) - Avg Wait: {res_noforecast.metrics.avg_wait:.2f}s | Util: {res_noforecast.metrics.gpu_utilization:.1%}")
else:
    print("Ablation checkpoint not found; run train_ppo with --no-forecast to generate.")


## 9. Key Findings & Architectural Conclusions
1. **Interactive Training in Notebooks**: By keeping the environment, cluster models, and algorithms in `src/`, notebooks can execute clean, interactive training loops and persist trained models to `artifacts/`.
2. **Guaranteed Placement Safety**: Action masking mathematically eliminates placement violations during PPO training and inference.
3. **Reproducibility**: Experiments use fixed random seeds and logged hyperparameters for deterministic comparison.
